# MobileNetV2 Knowledge Distillation for Weather Classification

In MH2, our custom 2-layer CNN student (23.9M params) only reached 91.18% — worse than both teachers. The problem was **architectural**: a naive Conv→Conv→FC lacks the inductive biases of pretrained models.

**This notebook tests a better student: MobileNetV2 (pretrained, 3.4M params)**
- Pretrained on ImageNet → already knows visual features
- Inverted residual blocks → efficient feature extraction
- 7x smaller than the original student, 42x smaller than the ensemble
- Designed for mobile/edge deployment

**Run on Google Colab with T4 GPU.** Export as `.ipynb` from Databricks.

In [0]:
!pip install -q torchsummary
!wget -q http://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Multi-classWeatherDataset.zip
!unzip -qq -o Multi-classWeatherDataset.zip

import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import time, copy, os, tempfile, itertools
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
from torchsummary import summary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")

In [0]:
# ==================== Transforms (with augmentation for training) ====================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==================== Dataset ====================
base_dataset = datasets.ImageFolder(root='Multi-class Weather Dataset')
class_names = base_dataset.classes
print(f"Classes: {class_names}, Total: {len(base_dataset)}")

train_size = int(0.70 * len(base_dataset))
val_size = int(0.15 * len(base_dataset))
test_size = len(base_dataset) - train_size - val_size

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, base_ds, indices, transform):
        self.samples = [(base_ds.samples[i][0], base_ds.samples[i][1]) for i in indices]
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

generator = torch.Generator().manual_seed(42)
idx = torch.randperm(len(base_dataset), generator=generator).tolist()
train_dataset = TransformSubset(base_dataset, idx[:train_size], train_transform)
val_dataset = TransformSubset(base_dataset, idx[train_size:train_size+val_size], test_transform)
test_dataset = TransformSubset(base_dataset, idx[train_size+val_size:], test_transform)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==================== Utilities ====================
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_loss, self.best_model = 0, None, None
    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

def evaluate_model(model, criterion, loader):
    model.eval()
    rloss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs); loss = criterion(out, labels)
            rloss += loss.item()*imgs.size(0)
            _, pred = torch.max(out, 1); total += labels.size(0); correct += (pred==labels).sum().item()
    return rloss/total, 100*correct/total

def get_predictions(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            _, p = torch.max(model(imgs.to(device)), 1)
            preds.extend(p.cpu().numpy()); labels.extend(lbl.numpy())
    return np.array(labels), np.array(preds)

print("Data loaded. Utilities ready.")

## Step 1: Train Teacher Models

We train both ResNet18 and VGG16 as teachers (same as MH2). The ensemble of these two models serves as the "teacher" for knowledge distillation.

We use the **fine-tuning approach** (unfreeze last conv block for VGG16) since it gave the best generalization in MH2.

In [0]:
def train_model(model, criterion, optimizers, train_loader, val_loader, max_epochs, name):
    """Train with early stopping, return model and test accuracy."""
    if not isinstance(optimizers, list): optimizers = [optimizers]
    es = EarlyStopping(patience=3)
    print(f"\nTraining {name}...")
    for epoch in range(max_epochs):
        model.train()
        rloss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            for o in optimizers: o.zero_grad()
            out = model(imgs); loss = criterion(out, labels); loss.backward()
            for o in optimizers: o.step()
            rloss += loss.item()*imgs.size(0)
            _, pred = torch.max(out, 1); total += labels.size(0); correct += (pred==labels).sum().item()
        tl, ta = rloss/total, 100*correct/total
        vl, va = evaluate_model(model, criterion, val_loader)
        print(f"  Epoch [{epoch+1:2d}/{max_epochs}] | Train: {tl:.4f}/{ta:.1f}% | Val: {vl:.4f}/{va:.1f}%")
        if es(vl, model):
            print(f"  Early stopping! Best val loss: {es.best_loss:.4f}")
            model.load_state_dict(es.best_model); break
    _, test_acc = evaluate_model(model, criterion, test_loader)
    print(f"  {name} Test Accuracy: {test_acc:.2f}%")
    return model, test_acc

# ==================== ResNet18 (FC-only) ====================
resnet = models.resnet18(weights='IMAGENET1K_V1')
for p in resnet.parameters(): p.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, 4)
resnet = resnet.to(device)
resnet, resnet_acc = train_model(
    resnet, nn.CrossEntropyLoss(), optim.Adam(resnet.fc.parameters(), lr=0.001),
    train_loader, val_loader, 20, "ResNet18")

# ==================== VGG16 (fine-tuned last conv block) ====================
vgg = models.vgg16(weights='IMAGENET1K_V1')
for p in vgg.parameters(): p.requires_grad = False
for p in vgg.features[24:].parameters(): p.requires_grad = True
vgg.classifier[6] = nn.Linear(4096, 4)
vgg = vgg.to(device)
opt_conv = optim.SGD(vgg.features[24:].parameters(), lr=0.0001, momentum=0.9)
opt_cls = optim.Adam(vgg.classifier[6].parameters(), lr=0.001)
vgg, vgg_acc = train_model(
    vgg, nn.CrossEntropyLoss(), [opt_conv, opt_cls],
    train_loader, val_loader, 20, "VGG16")

# ==================== Ensemble Accuracy ====================
def ensemble_acc_fn(models_list, loader):
    for m in models_list: m.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            avg = sum(F.softmax(m(imgs), dim=1) for m in models_list) / len(models_list)
            _, pred = torch.max(avg, 1)
            total += labels.size(0); correct += (pred==labels.to(device)).sum().item()
    return 100*correct/total

ens_acc = ensemble_acc_fn([resnet, vgg], test_loader)
print(f"\nEnsemble Test Accuracy: {ens_acc:.2f}%")
print(f"Teachers ready. Ensemble = {ens_acc:.2f}%")

## Step 2: MobileNetV2 Student — Knowledge Distillation

**Why MobileNetV2 is a better student than a custom 2-layer CNN:**

| Feature | Custom CNN (MH2) | MobileNetV2 |
|---------|-----------------|-------------|
| Parameters | 23.9M | **3.4M** (7x smaller) |
| Pretrained | No | **Yes** (ImageNet) |
| Architecture | 2 conv + FC | Inverted residual blocks |
| Inductive bias | None | Strong spatial hierarchy |
| MH2 test accuracy | 91.18% | **TBD** |

**MobileNetV2 key design features:**
- **Depthwise separable convolutions** — factorize standard convolution into depthwise + pointwise, dramatically reducing computation
- **Inverted residual blocks** — expand channels, apply depthwise conv, then compress back (opposite of ResNet)
- **Linear bottlenecks** — avoid information loss in the compression step

**Distillation setup (same as MH2):**
- Temperature T=3.0 (softer probabilities)
- Loss = 0.7 × KL_divergence(student_soft, teacher_soft) + 0.3 × CrossEntropy(student, true_labels)
- We freeze all pretrained layers and only train the new classifier head

In [0]:
# ==================== Build MobileNetV2 Student ====================
mobilenet = models.mobilenet_v2(weights='IMAGENET1K_V1')

# Freeze all pretrained layers
for p in mobilenet.parameters():
    p.requires_grad = False

# Replace classifier (1280 -> 4 classes)
mobilenet.classifier[1] = nn.Linear(1280, 4)
mobilenet = mobilenet.to(device)

mobile_params = sum(p.numel() for p in mobilenet.parameters())
mobile_trainable = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
print(f"MobileNetV2 total params: {mobile_params:,}")
print(f"MobileNetV2 trainable params: {mobile_trainable:,} (classifier only)")
print(f"vs MH2 Custom CNN: 23,907,908 params")
print(f"vs Ensemble teacher: ~145.5M params")
print(f"Compression: {145.5e6/mobile_params:.0f}x smaller than ensemble")

print("\nMobileNetV2 classifier:")
print(mobilenet.classifier)

# ==================== Distillation Training ====================
temperature = 3.0
alpha = 0.7
mobile_opt = optim.Adam(mobilenet.classifier[1].parameters(), lr=0.001)
criterion_hard = nn.CrossEntropyLoss()
early_stop_mobile = EarlyStopping(patience=3)

resnet.eval(); vgg.eval()
mobile_tl, mobile_vl, mobile_ta, mobile_va = [], [], [], []

print(f"\n{'='*60}")
print(f"Training MobileNetV2 via Knowledge Distillation (T={temperature})")
print(f"{'='*60}")

start = time.time()
for epoch in range(20):
    mobilenet.train()
    rloss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            t_logits = (resnet(imgs) + vgg(imgs)) / 2
            t_soft = F.softmax(t_logits / temperature, dim=1)
        s_logits = mobilenet(imgs)
        s_soft = F.log_softmax(s_logits / temperature, dim=1)
        loss_kd = F.kl_div(s_soft, t_soft, reduction='batchmean') * (temperature**2)
        loss_hard = criterion_hard(s_logits, labels)
        loss = alpha * loss_kd + (1-alpha) * loss_hard
        mobile_opt.zero_grad(); loss.backward(); mobile_opt.step()
        rloss += loss.item()*imgs.size(0)
        _, pred = torch.max(s_logits, 1); total += labels.size(0); correct += (pred==labels).sum().item()
    tl = rloss/total; ta = 100*correct/total
    vl, va = evaluate_model(mobilenet, criterion_hard, val_loader)
    mobile_tl.append(tl); mobile_vl.append(vl); mobile_ta.append(ta); mobile_va.append(va)
    print(f"Epoch [{epoch+1:2d}/20] | Train: {tl:.4f} / {ta:.2f}% | Val: {vl:.4f} / {va:.2f}%")
    if early_stop_mobile(vl, mobilenet):
        print(f"\nEarly stopping at epoch {epoch+1}!")
        mobilenet.load_state_dict(early_stop_mobile.best_model)
        break
mobile_time = time.time() - start

_, mobile_test_acc = evaluate_model(mobilenet, criterion_hard, test_loader)

# ==================== Plot ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(mobile_tl)+1)
ax1.plot(ep, mobile_tl, 'b-o', label='Train'); ax1.plot(ep, mobile_vl, 'r-s', label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
ax2.plot(ep, mobile_ta, 'b-o', label='Train'); ax2.plot(ep, mobile_va, 'r-s', label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.suptitle('MobileNetV2 Student — Knowledge Distillation', fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nMobileNetV2 Test Accuracy: {mobile_test_acc:.2f}%")
print(f"Knowledge transfer: {mobile_test_acc/ens_acc*100:.1f}% of ensemble teacher")
print(f"Training time: {mobile_time:.1f}s")

## Step 3: Original Custom CNN Student (for comparison)

We train the same 2-layer CNN from MH2 with identical distillation settings to get a fair head-to-head comparison with MobileNetV2.

In [0]:
# ==================== Custom CNN Student (same as MH2) ====================
class StudentCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 54 * 54, 128), nn.ReLU(), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x); x = x.view(x.size(0), -1); return self.classifier(x)

cnn_student = StudentCNN(num_classes=4).to(device)
cnn_params = sum(p.numel() for p in cnn_student.parameters())
cnn_opt = optim.Adam(cnn_student.parameters(), lr=0.001)
early_stop_cnn = EarlyStopping(patience=3)

cnn_tl, cnn_vl, cnn_ta, cnn_va = [], [], [], []

print(f"{'='*60}")
print(f"Training Custom CNN Student via Distillation (T={temperature})")
print(f"{'='*60}")

start = time.time()
for epoch in range(20):
    cnn_student.train()
    rloss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            t_logits = (resnet(imgs) + vgg(imgs)) / 2
            t_soft = F.softmax(t_logits / temperature, dim=1)
        s_logits = cnn_student(imgs)
        s_soft = F.log_softmax(s_logits / temperature, dim=1)
        loss_kd = F.kl_div(s_soft, t_soft, reduction='batchmean') * (temperature**2)
        loss_hard = criterion_hard(s_logits, labels)
        loss = alpha * loss_kd + (1-alpha) * loss_hard
        cnn_opt.zero_grad(); loss.backward(); cnn_opt.step()
        rloss += loss.item()*imgs.size(0)
        _, pred = torch.max(s_logits, 1); total += labels.size(0); correct += (pred==labels).sum().item()
    tl = rloss/total; ta = 100*correct/total
    vl, va = evaluate_model(cnn_student, criterion_hard, val_loader)
    cnn_tl.append(tl); cnn_vl.append(vl); cnn_ta.append(ta); cnn_va.append(va)
    print(f"Epoch [{epoch+1:2d}/20] | Train: {tl:.4f} / {ta:.2f}% | Val: {vl:.4f} / {va:.2f}%")
    if early_stop_cnn(vl, cnn_student):
        print(f"\nEarly stopping at epoch {epoch+1}!")
        cnn_student.load_state_dict(early_stop_cnn.best_model)
        break
cnn_time = time.time() - start
_, cnn_test_acc = evaluate_model(cnn_student, criterion_hard, test_loader)
print(f"\nCustom CNN Test Accuracy: {cnn_test_acc:.2f}%")

## Final Comparison: MobileNetV2 vs Custom CNN vs Teachers

Head-to-head comparison of both students against the teacher models and ensemble, including per-class accuracy, inference speed, and model size.

In [0]:
# ==================== Per-Class Comparison ====================
resnet_true, resnet_pred = get_predictions(resnet, test_loader)
vgg_true, vgg_pred = get_predictions(vgg, test_loader)
mobile_true, mobile_pred = get_predictions(mobilenet, test_loader)
cnn_true, cnn_pred = get_predictions(cnn_student, test_loader)

print(f"\n{'='*75}")
print(f"{'PER-CLASS ACCURACY COMPARISON':^75}")
print(f"{'='*75}")
print(f"{'Class':<10} {'ResNet18':<10} {'VGG16':<10} {'Ensemble':<10} {'CNN(MH2)':<10} {'MobileV2':<12} {'Winner':<10}")
print(f"{'-'*75}")
for i, cls in enumerate(class_names):
    mask = resnet_true == i; n = mask.sum()
    r = 100*(resnet_pred[mask]==i).sum()/n
    v = 100*(vgg_pred[mask]==i).sum()/n
    # Ensemble
    ens_correct = 0
    for m_true, m_pred_r, m_pred_v in zip(resnet_true[mask], resnet_pred[mask], vgg_pred[mask]):
        pass  # simplified
    m = 100*(mobile_pred[mask]==i).sum()/n
    c = 100*(cnn_pred[mask]==i).sum()/n
    best = max(r,v,m,c)
    w = 'MobileV2' if m==best else ('CNN' if c==best else ('ResNet18' if r==best else 'VGG16'))
    print(f"{cls:<10} {r:<10.1f} {v:<10.1f} {'--':<10} {c:<10.1f} {m:<12.1f} {w:<10}")
print(f"{'='*75}")

# ==================== Inference Speed ====================
def measure_speed(model, loader, runs=3):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.time()
            for imgs, _ in loader: _ = model(imgs.to(device))
            times.append(time.time()-t0)
    return np.mean(times)

mobile_speed = measure_speed(mobilenet, test_loader)
cnn_speed = measure_speed(cnn_student, test_loader)
resnet_speed = measure_speed(resnet, test_loader)
vgg_speed = measure_speed(vgg, test_loader)

# ==================== Final Summary ====================
print(f"\n{'='*80}")
print(f"{'FINAL COMPARISON: MobileNetV2 vs Custom CNN vs Teachers':^80}")
print(f"{'='*80}")
print(f"{'Model':<25} {'Test Acc':<11} {'Params':<15} {'Inference':<12} {'Pretrained':<10}")
print(f"{'-'*80}")
print(f"{'ResNet18 (teacher)':<25} {resnet_acc:<11.2f} {'11.2M':<15} {resnet_speed:.3f}s{'':5} {'Yes':<10}")
print(f"{'VGG16 (teacher)':<25} {vgg_acc:<11.2f} {'134.3M':<15} {vgg_speed:.3f}s{'':5} {'Yes':<10}")
print(f"{'Ensemble (teacher)':<25} {ens_acc:<11.2f} {'~145.5M':<15} {'--':<12} {'Yes':<10}")
print(f"{'Custom CNN (MH2)':<25} {cnn_test_acc:<11.2f} {f'{cnn_params/1e6:.1f}M':<15} {cnn_speed:.3f}s{'':5} {'No':<10}")
print(f"{'MobileNetV2 (NEW)':<25} {mobile_test_acc:<11.2f} {f'{mobile_params/1e6:.1f}M':<15} {mobile_speed:.3f}s{'':5} {'Yes':<10}")
print(f"{'='*80}")

# Improvement analysis
improvement = mobile_test_acc - cnn_test_acc
print(f"\nMobileNetV2 vs Custom CNN: {improvement:+.2f}% accuracy")
print(f"MobileNetV2 vs Custom CNN: {cnn_params/mobile_params:.1f}x fewer parameters")
print(f"Knowledge transfer: {mobile_test_acc/ens_acc*100:.1f}% of ensemble accuracy")

# ==================== Bar Chart ====================
fig, ax = plt.subplots(figsize=(10, 6))
names = ['ResNet18', 'VGG16', 'Ensemble', 'CNN Student\n(MH2)', 'MobileNetV2\n(NEW)']
accs = [resnet_acc, vgg_acc, ens_acc, cnn_test_acc, mobile_test_acc]
colors = ['#1f77b4', '#ff7f0e', '#9467bd', '#d62728', '#2ca02c']
bars = ax.bar(names, accs, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=95, color='red', ls='--', alpha=0.7, label='95% target')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Knowledge Distillation: MobileNetV2 vs Custom CNN', fontsize=14, fontweight='bold')
ax.set_ylim(min(accs)-5, 102)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

# ==================== Save & Download MobileNetV2 ====================
from google.colab import files

model_path = 'mobilenetv2_distilled_weather.pt'
torch.save({
    'model_state_dict': mobilenet.state_dict(),
    'class_names': class_names,
    'accuracy': mobile_test_acc,
    'architecture': 'MobileNetV2 (pretrained) + distilled from ResNet18+VGG16 ensemble',
    'params': mobile_params,
    'input_size': (3, 224, 224),
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
}, model_path)

file_size = os.path.getsize(model_path) / (1024*1024)
print(f"\nModel saved: {model_path} ({file_size:.1f} MB)")
print(f"Accuracy: {mobile_test_acc:.2f}%, Params: {mobile_params:,}")
print(f"Downloading...")
files.download(model_path)

## Observations

### Why MobileNetV2 should be a better student:
1. **Pretrained features** — already understands visual patterns from 1.2M ImageNet images, so soft labels just need to guide the classifier, not teach vision from scratch
2. **3.4M params vs 23.9M** — 7x smaller because efficient depthwise separable convolutions replace the massive FC layer (64×54×54 → 128 = 23.9M in the custom CNN)
3. **Better inductive biases** — inverted residual blocks preserve spatial information that a 2-layer Conv→FC architecture loses
4. **Production-ready** — designed by Google specifically for mobile/edge deployment

### Comparison with MH2 results:
- MH2 Custom CNN: 91.18% (23.9M params) — limited by architecture
- MH2 Pruned VGG16: 95.88% (40.3M active) — best MH2 model
- MobileNetV2: **Check results above** — should approach or exceed ensemble accuracy with fraction of parameters

### Key takeaway:
Knowledge distillation works best when the student has **sufficient architectural quality** (pretrained features + efficient blocks), not just parameter count. A 3.4M pretrained model can outperform a 23.9M untrained one because **architecture > size**.